# Code 10.7.2: Collecting rollouts

Runs each of several environments for `n_steps` with the current policy, storing everything the update will need, including the old log-probabilities and values. An episode that ends by the time limit (truncation) gets the discounted value of its final state added to its last reward.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 10.7.1
import numpy as np

import torch


In [ ]:
def collect_rollout(envs, obs, model, n_steps, gamma, stats):
    """Run every environment for n_steps with the current policy.

    Returns a dict of tensors with a leading (n_steps, n_envs) shape and the
    observations to continue from. Finished-episode returns go into stats.
    """
    n_envs = len(envs)
    buf = {k: [] for k in ("obs", "act", "logp", "rew", "term", "val")}
    for _ in range(n_steps):
        s = torch.as_tensor(np.array(obs), dtype=torch.float32)
        with torch.no_grad():
            d = model.dist(s)
            a = d.sample()
            logp, v = d.log_prob(a), model.value(s)
        rew, term = np.zeros(n_envs, np.float32), np.zeros(n_envs, np.float32)
        for i, env in enumerate(envs):
            s2, r, terminated, truncated, _ = env.step(int(a[i]))
            stats["ep_ret"][i] += r
            if truncated and not terminated:
                # A time limit is not a real ending: bootstrap from V(s_final).
                with torch.no_grad():
                    r += gamma * model.value(torch.as_tensor(s2, dtype=torch.float32)).item()
            if terminated or truncated:
                stats["finished"].append(stats["ep_ret"][i])
                stats["ep_ret"][i] = 0.0
                s2, _ = env.reset()
            rew[i], term[i] = r, float(terminated or truncated)
            obs[i] = s2
        for k, x in zip(buf, (s, a, logp, torch.as_tensor(rew), torch.as_tensor(term), v)):
            buf[k].append(x)
    return {k: torch.stack(x) for k, x in buf.items()}, obs
